In [1]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as wg
import xarray as xr
import datetime as dt

In [2]:
ds = xr.open_dataset('output.nc')

In [3]:
ds

<xarray.Dataset> Size: 494kB
Dimensions:                               (
                                           FullWaveDirectionalSpectra_Frequency: 34,
                                           FullWaveDirectionalSpectra_Direction: 36,
                                           time: 49)
Coordinates:
  * FullWaveDirectionalSpectra_Frequency  (FullWaveDirectionalSpectra_Frequency) float64 272B ...
  * FullWaveDirectionalSpectra_Direction  (FullWaveDirectionalSpectra_Direction) float64 288B ...
  * time                                  (time) datetime64[ns] 392B 1970-01-...
Data variables:
    EnergySpectra                         (time, FullWaveDirectionalSpectra_Frequency) float64 13kB ...
    FullWaveDirectionalSpectra_Energy     (time, FullWaveDirectionalSpectra_Direction, FullWaveDirectionalSpectra_Frequency) float64 480kB ...

In [4]:
freq = ds.FullWaveDirectionalSpectra_Frequency.values         # частоты [Гц]
dirs = ds.FullWaveDirectionalSpectra_Direction.values         # направления [град]
time = ds.time.values                                         # время
energy_omni = ds.EnergySpectra.values                         # размер (time, freq)
energy_dir = ds.FullWaveDirectionalSpectra_Energy.values      # размер (time, dir, freq)

def format_time(t):
    if isinstance(t, np.datetime64):
        full_str = np.datetime_as_string(t, unit='s')
        return full_str.split('T')[-1]
    else:
        total_seconds = int(round(t))
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        seconds = total_seconds % 60
        return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

time_slider = wg.IntSlider(min=0, max=len(time)-1, step=1, value=0,
                           description='Time index',
                           continuous_update=True,
                           layout=wg.Layout(width='600px'))

@wg.interact(time_index=time_slider)
def plot_spectrum(time_index):
    E_omni = energy_omni[time_index, :]
    E_dir = energy_dir[time_index, :, :]

    fig = plt.figure(figsize=(12,5))
    plt.rc('lines', linewidth=1.2)
    plt.rcParams.update({'font.size': 14})
    fig.patch.set_facecolor('white')

    gs = fig.add_gridspec(1,2)
    gs.update(hspace=0.05, wspace=0.3)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1], polar=True)

    # Левый график
    ax1.plot(freq, E_omni, color='tab:blue')
    ax1.plot(freq, E_omni, color='tab:blue', marker='o', markersize=4, linestyle='-', linewidth=1)
    ax1.set_xlabel('$f$, Hz')
    ax1.set_ylabel('$E(f)$, $m^2/Hz$')
    ax1.set_title(f'Omnidirectional spectrum at {format_time(time[time_index])}')
    ax1.grid(True)

    # Правый график
    theta = np.radians(dirs)
    r = freq
    levels=np.linspace(np.mean(energy_dir[time_index]),
                                       np.amax(energy_dir[time_index]), 15)

    cmap = plt.cm.RdBu_r
    neg_inf = cmap(-2)
    cmap.set_bad(neg_inf)
    cmap.set_under('w')
    cntf = ax2.contourf(theta, r, E_dir.T, vmin=0.0001, levels=levels, cmap=cmap, extend='both')
    plt.colorbar(cntf, ax=ax2, label='$E(f,\\theta)$, $m^2/(Hz \\cdot rad)$')

    ax2.set_theta_zero_location('N')
    ax2.set_theta_direction(-1)
    ax2.grid(True)

    plt.show()

interactive(children=(IntSlider(value=0, description='Time index', layout=Layout(width='600px'), max=48), Outp…